# Young Investor Behavioural Analysis & Segmentation
## RBC Capstone Project | Team 24

This notebook identifies and profiles young investors using demographic and behavioural
data, then applies K-Prototypes clustering to segment them into actionable groups.

**Set New Condition 2 upper bound: 65 (standard retirement age)**

In [ ]:
!pip install kmodes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
from kmodes.kprototypes import KPrototypes
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings("ignore")

## 1. Data Loading

In [ ]:
core = pd.read_parquet("Broadridge_Core_All.parquet")
identity = pd.read_parquet("Identity_All.parquet")

account_cols = [
    "PrimaryOwnerPartyId", "FinancialAccountId", "OpenDate",
    "InvestmentObjectives", "InvestmentTimeHorizon", "LiquidityNeeds",
    "RiskTolerance", "SourceofFunds"
]
account = core[account_cols].drop_duplicates(subset="FinancialAccountId")
party = identity[["PartyId", "PartyAge"]].drop_duplicates(subset="PartyId")

print(f"Accounts loaded: {len(account)}")
print(f"Parties loaded: {len(party)}")

## 2. Age Statistics

In [ ]:
party["PartyAge_num"] = pd.to_numeric(
    party["PartyAge"].replace("NULL", pd.NA), errors="coerce"
)

average_age = party["PartyAge_num"].dropna().mean()
median_age = party["PartyAge_num"].dropna().median()
std_age = party["PartyAge_num"].dropna().std()
young_threshold = average_age - std_age

print(f"Average age of investors: {average_age:.2f}")
print(f"Median age of investors: {median_age:.2f}")
print(f"Standard deviation: {std_age:.2f}")
print(f"Young investor cutoff age (mean minus 1 std): {young_threshold:.2f}")

## 3. Identifying Young Investors

In [ ]:
account["OpenDate"] = pd.to_datetime(account["OpenDate"])

df = pd.merge(account, party, left_on="PrimaryOwnerPartyId", right_on="PartyId")

current_year = datetime.now().year
five_years_ago = current_year - 5

print(f"Merged dataset: {len(df)} rows")
print(f"Current year: {current_year}, five year lookback starts: {five_years_ago}")

In [ ]:
# Condition 1: accounts opened in the last 5 years by investors whose age
# falls below one standard deviation from the mean (younger than ~33).

condition1 = (
    (df["OpenDate"].dt.year >= five_years_ago) &
    (df["PartyAge_num"] < young_threshold)
)

# Condition 2: accounts opened in the last 5 years by investors whose age
# is between the young threshold and 65 (standard retirement age), AND who
# have low risk tolerance, suggesting behaviourally conservative profiles
# that may resemble newer or less experienced investors.
#
# CHANGED FROM ORIGINAL: added an upper bound of 65 to condition2.
# The original code only had (PartyAge >= young_threshold) with no upper
# limit, which unintentionally captured investors well above the mean,
# including retirees (average age 67 in the uncapped version). Low risk
# tolerance correlates strongly with older age in this dataset (Low Risk
# group mean age = 74), so without the cap the filter pulled in elderly
# conservative investors rather than "behaviourally young" ones.
# Using the standard retirement age (65) as the upper bound provides a
# natural demographic boundary. Investors below 65 are still in their
# working and accumulation phase, while those above are typically in
# retirement and distribution mode.

condition2 = (
    (df["OpenDate"].dt.year >= five_years_ago) &
    (df["PartyAge_num"] >= young_threshold) &
    (df["PartyAge_num"] < 65) &
    (df["RiskTolerance"].isin(["Low Risk"]))
)

young_investors = df[condition1 | condition2].copy()

print(f"Number of young investors (accounts): {len(young_investors)}")
print(f"Unique parties: {young_investors['PartyId'].nunique()}")
print(f"Mean age: {young_investors['PartyAge_num'].mean():.2f}")
print(f"")
print(f"Condition 1 (age < {young_threshold:.0f}): {condition1.sum()} accounts")
print(f"Condition 2 ({young_threshold:.0f} to 65, Low Risk): {condition2.sum()} accounts")

In [ ]:
print("Age distribution of young investors:")
print(young_investors["PartyAge_num"].describe())

## 4. Source of Funds Classification

In [ ]:
young_investors["SourceofFunds"].value_counts()

In [ ]:
earned_sources = [
    "Savings",
    "Wages/Income",
    "Investment Income",
    "Business/Self-employment",
    "Sale of Asset",
    "Settlement"
]

inherited_sources = [
    "Gift/Inheritance",
    "Transfer",
    "Rollover"
]

young_investors["WealthType"] = young_investors["SourceofFunds"].apply(
    lambda x: "Earned" if x in earned_sources else "Inherited"
)

young_investors["WealthType"].value_counts()

In [ ]:
print("Investment Objectives:")
print(young_investors["InvestmentObjectives"].unique())
print("")
print("Investment Time Horizon:")
print(young_investors["InvestmentTimeHorizon"].unique())
print("")
print("Liquidity Needs:")
print(young_investors["LiquidityNeeds"].unique())

## 5. Behavioural Analysis: Earned vs Inherited

In [ ]:
objective_chart = pd.crosstab(
    young_investors["InvestmentObjectives"],
    young_investors["WealthType"]
)

objective_chart["Total"] = objective_chart.sum(axis=1)
objective_chart = objective_chart.sort_values(by="Total", ascending=False).drop(columns="Total")

objective_chart.plot(kind="bar", figsize=(8, 5), color=["red", "orange"])
plt.title("Investment Objectives: Earned vs Inherited Young Investors")
plt.ylabel("Number of Investors")
plt.xlabel("Investment Objective")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Wealth Type")
plt.tight_layout()
plt.show()

**Interpretation: Investment Objectives**

Growth is the dominant objective across both wealth types. Balanced Growth is nearly equal between the two groups, meaning the preference for a moderate, diversified strategy is not influenced by wealth source. Preservation of Principal and Speculation are negligible, confirming that young investors are generally not in capital protection or speculative mode.

In [ ]:
horizon_chart = pd.crosstab(
    young_investors["InvestmentTimeHorizon"],
    young_investors["WealthType"]
)

horizon_chart["Total"] = horizon_chart.sum(axis=1)
horizon_chart = horizon_chart.sort_values(by="Total", ascending=False).drop(columns="Total")

horizon_chart.plot(kind="bar", figsize=(8, 5), color=["red", "orange"])
plt.title("Investment Time Horizon: Earned vs Inherited Young Investors")
plt.ylabel("Number of Investors")
plt.xlabel("Investment Time Horizon")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Wealth Type")
plt.tight_layout()
plt.show()

**Interpretation: Investment Time Horizon**

The distribution is extremely skewed toward "12 years or more" (~2,770 accounts), which
is expected for a young investor population that has decades ahead before retirement.
Among the shorter horizon categories (Less than 1 year, 1 to 3 years), Inherited investors
show slightly higher counts than Earned investors, possibly indicating that some recipients
of transferred wealth have nearer term liquidity needs or have not yet committed to a
long term investment plan. The very small numbers in the 4 to 11 year range suggest that
young investors tend to think in binary terms: either very long term or very short term,
with little in between.

In [ ]:
liquidity_chart = pd.crosstab(
    young_investors["LiquidityNeeds"],
    young_investors["WealthType"]
)

liquidity_chart["Total"] = liquidity_chart.sum(axis=1)
liquidity_chart = liquidity_chart.sort_values(by="Total", ascending=False).drop(columns="Total")

liquidity_chart.plot(kind="bar", figsize=(8, 5), color=["red", "orange"])
plt.title("Liquidity Needs: Earned vs Inherited Young Investors")
plt.ylabel("Number of Investors")
plt.xlabel("Liquidity Needs")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Wealth Type")
plt.tight_layout()
plt.show()

**Interpretation: Liquidity Needs**

The vast majority of young investors report liquidity needs of less than 1,000, meaning they do not expect to withdraw significant cash in the near term. This is a positive signal for RBC, as it suggests these assets are relatively "sticky" and advisors have a meaningful time window to build relationships and offer guidance before any large withdrawals occur. The 1,000 to 9,999 tier is the only other notable group, split roughly evenly between Earned and Inherited. Higher liquidity tiers are minimal, reinforcing that this population is in accumulation mode, not distribution mode.

## 6. Supplementary Analysis

In [ ]:
# visualize the age distribution by condition to confirm the two
# sub populations are reasonable after the condition2 correction.

young_investors["Segment"] = "Condition 2 (33 to 65, Low Risk)"
young_investors.loc[
    young_investors["PartyAge_num"] < young_threshold, "Segment"
] = "Condition 1 (Age < 33)"

c1_ages = young_investors.loc[
    young_investors["Segment"].str.startswith("Condition 1"), "PartyAge_num"
].dropna()
c2_ages = young_investors.loc[
    young_investors["Segment"].str.startswith("Condition 2"), "PartyAge_num"
].dropna()

plt.figure(figsize=(8, 5))
plt.hist(
    [c1_ages, c2_ages],
    bins=range(15, 70, 2),
    color=["#2196F3", "#FF9800"],
    label=[f"Condition 1: Age < {young_threshold:.0f} (n={len(c1_ages)})",
           f"Condition 2: {young_threshold:.0f} to 65 + Low Risk (n={len(c2_ages)})"],
    stacked=True,
    edgecolor="white"
)
plt.axvline(x=young_threshold, color="red", linestyle="--",
            label=f"Lower cutoff ({young_threshold:.0f})")
plt.axvline(x=65, color="green", linestyle="--",
            label="Upper cutoff (65)")
plt.title("Age Distribution of Young Investors by Condition")
plt.xlabel("Age")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Condition 1 mean age: {c1_ages.mean():.1f}")
print(f"Condition 2 mean age: {c2_ages.mean():.1f}")

**Interpretation: Age Distribution by Condition**

This chart validates the two condition filtering logic. Condition 1 (blue, n=2,552)
captures investors under age 33 and shows a roughly normal distribution peaking around
ages 28 to 32, with a mean of 26.4. Condition 2 (orange, n=592) captures investors
aged 33 to 65 who have Low Risk tolerance, and shows a relatively flat distribution with
a slight concentration in the 58 to 64 age range, with a mean of 52.8. The two populations
are clearly distinct with no overlap, confirming that condition2 is adding a genuinely
different sub population rather than duplicating condition1. The 65 year upper cutoff
(green dashed line) successfully excludes retirees while capturing pre retirement
conservative investors.

In [ ]:
# risk tolerance breakdown by wealth type

risk_chart = pd.crosstab(
    young_investors["RiskTolerance"],
    young_investors["WealthType"]
)

risk_order = ["Maximum Risk", "High Risk", "Moderate Risk", "Low Risk", "Minimal Risk"]
risk_chart = risk_chart.reindex([r for r in risk_order if r in risk_chart.index])

risk_chart.plot(kind="barh", figsize=(8, 5), color=["red", "orange"])
plt.title("Risk Tolerance: Earned vs Inherited Young Investors")
plt.xlabel("Number of Investors")
plt.ylabel("Risk Tolerance")
plt.legend(title="Wealth Type")
plt.tight_layout()
plt.show()

print("Risk Tolerance by Wealth Type:")
print(pd.crosstab(
    young_investors["RiskTolerance"],
    young_investors["WealthType"],
    margins=True
))

**Interpretation: Risk Tolerance**

Moderate Risk is the most common category (1,403 accounts), with Earned investors
outnumbering Inherited by about 200. High Risk is the second largest (1,016 accounts),
again with Earned slightly higher. Low Risk ranks third (652 accounts), largely driven
by the Condition 2 sub population (pre retirement conservative investors). The overall
pattern shows that Earned wealth investors tend to have slightly higher risk appetite
than Inherited wealth investors. This makes intuitive sense: people who accumulated wealth
through their own efforts may feel more confident in their investment judgment, while
those who received wealth may be more cautious about preserving what they were given.
Maximum Risk and Minimal Risk are both negligible.

In [ ]:
# annual income and total net worth by wealth type.
# Income and net worth data are pulled from Identity_All (party level).

identity_demo = identity[["PartyId", "AnnualIncome", "TotalNetWorth"]].drop_duplicates(
    subset="PartyId"
)
young_investors = young_investors.merge(identity_demo, on="PartyId", how="left")

income_order = [
    "Less than $50,000", "$50,000 - $99,999", "$100,000 - $199,999",
    "$200,000 - $299,999", "$300,000 - $399,999", "$400,000 - $499,999",
    "$500,000 - $749,999", "$750,000 - $999,999", "$1,000,000+"
]

inc_chart = pd.crosstab(young_investors["AnnualIncome"], young_investors["WealthType"])
inc_chart = inc_chart.drop("NULL", errors="ignore")
inc_chart = inc_chart.reindex([i for i in income_order if i in inc_chart.index])

inc_chart.plot(kind="barh", figsize=(8, 5), color=["red", "orange"])
plt.title("Annual Income: Earned vs Inherited Young Investors")
plt.xlabel("Number of Investors")
plt.ylabel("Annual Income")
plt.legend(title="Wealth Type")
plt.tight_layout()
plt.show()

print("Annual Income by Wealth Type:")
print(pd.crosstab(
    young_investors["AnnualIncome"],
    young_investors["WealthType"],
    margins=True
).drop("NULL", errors="ignore"))

**Interpretation: Annual Income**

The income distribution is heavily concentrated at the lower end: Less than 50,000 and 50,000 to 99,999 together account for about 78% of all young investors, consistent with a population in the early stages of their careers.
Earned investors outnumber Inherited investors in nearly every income bracket, which is expected since Earned investors by definition have active income sources. One notable detail is in the $200K to $300K range where Inherited (66) slightly exceeds Earned (65) and at $500K+ where Earned (28) significantly exceeds Inherited (12), suggesting that the very highest earning young investors tend to invest their own money rather than relying on inherited wealth.

In [ ]:
# total net worth by wealth type

nw_order = [
    "Less than $100,000", "$100,000 - $249,999", "$250,000 - $499,999",
    "$500,000 - $999,999", "$1,000,000 - $2,999,999", "$3,000,000+"
]

nw_chart = pd.crosstab(young_investors["TotalNetWorth"], young_investors["WealthType"])
nw_chart = nw_chart.drop("NULL", errors="ignore")
nw_chart = nw_chart.reindex([n for n in nw_order if n in nw_chart.index])

nw_chart.plot(kind="barh", figsize=(8, 5), color=["red", "orange"])
plt.title("Total Net Worth: Earned vs Inherited Young Investors")
plt.xlabel("Number of Investors")
plt.ylabel("Total Net Worth")
plt.legend(title="Wealth Type")
plt.tight_layout()
plt.show()

print("Total Net Worth by Wealth Type:")
print(pd.crosstab(
    young_investors["TotalNetWorth"],
    young_investors["WealthType"],
    margins=True
).drop("NULL", errors="ignore"))

**Interpretation: Total Net Worth**

While Less than 100000 is again the largest category (1,727 accounts), the high net worth tiers reveal an important pattern. In the $1M to $3M bracket, Inherited (128) slightly exceeds Earned (122), and in the $3M+ bracket, Inherited (103) substantially exceeds Earned (74). This is intuitive is that it's difficult for young people to accumulate multi million dollar net worth through earnings alone in a short time, but inheritance and wealth transfers can create this result immediately. For RBC, this means that the highest value young clients are disproportionately Inherited wealth holders, precisely the population most at risk of leaving during the Great Wealth Transfer. These are the clients where proactive advisor engagement matters most.

## 7. Export Behavioural Data

In [ ]:
young_subset = young_investors[[
    "PartyId", "FinancialAccountId", "PartyAge_num",
    "InvestmentObjectives", "InvestmentTimeHorizon", "RiskTolerance",
    "SourceofFunds", "LiquidityNeeds", "WealthType", "Segment"
]]

young_subset.to_csv("young_investors_behavior.csv", index=False)
print(f"Exported {len(young_subset)} rows to young_investors_behavior.csv")
print(young_subset.head())

## 8. Young Investor Segmentation (K-Prototypes Clustering)

We use K-Prototypes clustering, which handles mixed data types (numerical and categorical)
simultaneously. This allows us to include both continuous features like age and ordinal
income, as well as categorical features like investment objectives and wealth type.

K-Prototypes extends K-Means by combining Euclidean distance for numerical features
with Hamming distance for categorical features, making it suitable for our mixed dataset.

### 8.1 Feature Engineering

We select 7 features across three dimensions:
- **Demographic**: age, annual income (ordinal), total net worth (ordinal)
- **Investment profile**: risk tolerance (ordinal)
- **Behavioural / categorical**: investment objectives, time horizon, wealth type (earned vs inherited)

Income, net worth, and risk tolerance are encoded as ordinal integers to preserve their
natural ordering. All numerical features are then scaled to [0, 1] so that no single
feature dominates the distance calculation.

In [ ]:
income_map = {
    "NULL": 0, "Less than $50,000": 1, "$50,000 - $99,999": 2,
    "$100,000 - $199,999": 3, "$200,000 - $299,999": 4,
    "$300,000 - $399,999": 5, "$400,000 - $499,999": 6,
    "$500,000 - $749,999": 7, "$750,000 - $999,999": 8, "$1,000,000+": 9
}
nw_map = {
    "NULL": 0, "Less than $100,000": 1, "$100,000 - $249,999": 2,
    "$250,000 - $499,999": 3, "$500,000 - $999,999": 4,
    "$1,000,000 - $2,999,999": 5, "$3,000,000+": 6
}
risk_map = {
    "Minimal Risk": 1, "Low Risk": 2, "Moderate Risk": 3,
    "High Risk": 4, "Maximum Risk": 5
}

young_investors["Income_ord"] = young_investors["AnnualIncome"].map(income_map).fillna(0).astype(int)
young_investors["NetWorth_ord"] = young_investors["TotalNetWorth"].map(nw_map).fillna(0).astype(int)
young_investors["Risk_ord"] = young_investors["RiskTolerance"].map(risk_map).fillna(3).astype(int)

# Separate numerical and categorical columns
num_cols = ["PartyAge_num", "Income_ord", "NetWorth_ord", "Risk_ord"]
cat_cols = ["InvestmentObjectives", "InvestmentTimeHorizon", "WealthType"]

feature_df = young_investors[num_cols + cat_cols].copy()

# Scale numerical features to [0, 1]
scaler = MinMaxScaler()
feature_df[num_cols] = scaler.fit_transform(feature_df[num_cols])

feature_matrix = feature_df.values
cat_indices = [feature_df.columns.get_loc(c) for c in cat_cols]

print(f"Feature matrix: {feature_matrix.shape[0]} samples x {feature_matrix.shape[1]} features")
print(f"Numerical features (indices 0 to 3): {num_cols}")
print(f"Categorical features (indices {cat_indices}): {cat_cols}")

### 8.2 Choosing k: Elbow Method

We run K-Prototypes for k = 2 through 8 and plot the total cost (sum of numerical
distance + categorical dissimilarity). The "elbow" where the cost reduction rate
drops off suggests a good number of clusters.

In [ ]:
costs = []
k_range = range(2, 9)

for k in k_range:
    kp = KPrototypes(n_clusters=k, init="Cao", n_init=3, random_state=42, verbose=0)
    kp.fit(feature_matrix, categorical=cat_indices)
    costs.append(kp.cost_)
    print(f"k={k}: cost={kp.cost_:.2f}")

plt.figure(figsize=(8, 5))
plt.plot(list(k_range), costs, "bo-", linewidth=2, markersize=8)
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Cost")
plt.title("Elbow Method for K-Prototypes")
plt.xticks(list(k_range))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nCost reduction at each step:")
for i in range(1, len(costs)):
    reduction = (costs[i-1] - costs[i]) / costs[i-1] * 100
    print(f"  k={i+1} to k={i+2}: {reduction:.1f}% reduction")

We select **k = 4** as the final model because it
produces clusters that are both statistically distinct and commercially interpretable,
while the marginal improvement beyond k = 4 comes with diminishing returns.

### 8.3 Final Clustering (k = 4)

In [ ]:
kp_final = KPrototypes(n_clusters=4, init="Cao", n_init=5, random_state=42, verbose=0)
labels = kp_final.fit_predict(feature_matrix, categorical=cat_indices)
young_investors["Cluster"] = labels

print("Cluster sizes:")
for c_id in sorted(young_investors["Cluster"].unique()):
    n = (young_investors["Cluster"] == c_id).sum()
    print(f"  Cluster {c_id}: {n} ({100*n/len(young_investors):.1f}%)")

### 8.4 Cluster Profiles

In [ ]:
# Build a summary table of key metrics per cluster
summary_rows = []

for c_id in sorted(young_investors["Cluster"].unique()):
    cl = young_investors[young_investors["Cluster"] == c_id]
    summary_rows.append({
        "Cluster": c_id,
        "n": len(cl),
        "% of Total": f"{100*len(cl)/len(young_investors):.1f}%",
        "Avg Age": f"{cl['PartyAge_num'].mean():.1f}",
        "Earned %": f"{100*(cl['WealthType']=='Earned').sum()/len(cl):.0f}%",
        "Top Risk": cl["RiskTolerance"].value_counts().index[0],
        "Top Objective": cl["InvestmentObjectives"].value_counts().index[0],
        "Top Income": cl["AnnualIncome"].value_counts().head(1).index[0],
        "Top Net Worth": cl["TotalNetWorth"].value_counts().head(1).index[0],
    })

summary_table = pd.DataFrame(summary_rows)
summary_table

In [ ]:
# Detailed printout for each cluster
for c_id in sorted(young_investors["Cluster"].unique()):
    cl = young_investors[young_investors["Cluster"] == c_id]
    print(f"{'='*60}")
    print(f"Cluster {c_id} (n={len(cl)}, {100*len(cl)/len(young_investors):.1f}%)")
    print(f"{'='*60}")
    print(f"  Age:        mean={cl['PartyAge_num'].mean():.1f}, median={cl['PartyAge_num'].median():.0f}")
    print(f"  WealthType: Earned {100*(cl['WealthType']=='Earned').sum()/len(cl):.0f}% | Inherited {100*(cl['WealthType']=='Inherited').sum()/len(cl):.0f}%")
    print(f"  Risk:       {cl['RiskTolerance'].value_counts().to_dict()}")
    print(f"  Objectives: {cl['InvestmentObjectives'].value_counts().to_dict()}")
    print(f"  Horizon:    {cl['InvestmentTimeHorizon'].value_counts().head(3).to_dict()}")
    print(f"  Income:     {cl['AnnualIncome'].value_counts().head(3).to_dict()}")
    print(f"  Net Worth:  {cl['TotalNetWorth'].value_counts().head(3).to_dict()}")
    print(f"  Source:     {cl['SourceofFunds'].value_counts().head(3).to_dict()}")
    print()

### 8.5 Cluster Visualization

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'figure.titlesize': 18
})

fig, axes = plt.subplots(2, 2, figsize=(18, 13))
colors = ["#2196F3", "#F44336", "#FF9800", "#4CAF50"]

cluster_names = {
    0: "Steady Builders",
    1: "Aggressive Growers",
    2: "Passive Inheritors",
    3: "Established Conservatives"
}

# Panel 1: Age distribution by cluster
ax = axes[0, 0]
for c_id in range(4):
    cl_ages = young_investors[young_investors["Cluster"] == c_id]["PartyAge_num"]
    ax.hist(cl_ages, bins=range(15, 70, 2), alpha=0.5,
            label=cluster_names[c_id], color=colors[c_id])
ax.set_title("Age Distribution by Cluster", fontweight="bold")
ax.set_xlabel("Age")
ax.set_ylabel("Count")
ax.legend(loc="upper right")

# Panel 2: Risk Tolerance stacked by cluster
ax = axes[0, 1]
risk_order = ["Maximum Risk", "High Risk", "Moderate Risk", "Low Risk", "Minimal Risk"]
risk_ct = pd.crosstab(young_investors["RiskTolerance"], young_investors["Cluster"])
risk_ct = risk_ct.reindex([r for r in risk_order if r in risk_ct.index])
risk_ct = risk_ct.rename(columns=cluster_names)
risk_ct.plot(kind="barh", ax=ax, color=colors, stacked=True)
ax.set_title("Risk Tolerance by Cluster", fontweight="bold")
ax.set_xlabel("Number of Investors")
ax.set_ylabel("")
ax.legend(loc="lower right", title="Cluster")

# Panel 3: Investment Objectives stacked by cluster
ax = axes[1, 0]
obj_ct = pd.crosstab(young_investors["InvestmentObjectives"], young_investors["Cluster"])
obj_ct = obj_ct.rename(columns=cluster_names)
obj_ct.plot(kind="barh", ax=ax, color=colors, stacked=True)
ax.set_title("Investment Objectives by Cluster", fontweight="bold")
ax.set_xlabel("Number of Investors")
ax.set_ylabel("")
ax.legend(loc="lower right", title="Cluster")

# Panel 4: Wealth Type composition by cluster
ax = axes[1, 1]
comp_data = []
for c_id in range(4):
    cl = young_investors[young_investors["Cluster"] == c_id]
    comp_data.append({
        "Cluster": cluster_names[c_id],
        "Earned": (cl["WealthType"] == "Earned").sum(),
        "Inherited": (cl["WealthType"] == "Inherited").sum()
    })
comp_df = pd.DataFrame(comp_data).set_index("Cluster")
comp_df.plot(kind="bar", ax=ax, color=["#2196F3", "#FF9800"], rot=25)
ax.set_title("Wealth Type Composition by Cluster", fontweight="bold")
ax.set_ylabel("Number of Investors")
ax.set_xlabel("")
ax.legend()

plt.suptitle("Young Investor Segmentation: K-Prototypes (k=4)",
             fontsize=18, fontweight="bold", y=0.995)
plt.tight_layout()
plt.savefig("cluster_visualization_hd.png", dpi=300, bbox_inches="tight")
plt.show()

**Age Distribution by Cluster**

Clusters 0, 1, and 2 have nearly identical age distributions, all concentrated between ages 18 and 32 with peaks around 28 to 32. This indicates that the algorithm distinguishes these three groups not by age, but by other features such as risk tolerance, investment objectives, and wealth source. Cluster 3 stands apart on the right side of the chart, spanning ages 35 to 64 with a visible concentration in the late 50s to early 60s. These are the pre retirement conservative investors captured by Condition 2.

**Risk Tolerance by Cluster**

This chart most clearly reveals the core differences between the four clusters. Moderate Risk is dominated by Clusters 0 and 2, showing that both groups share a similar middle ground risk appetite. High Risk is almost entirely Cluster 1, confirming that this group is defined by its aggressive risk profile. Low Risk is predominantly Cluster 3, aligning with the older, wealthier, and more conservative profile of that group. In summary, risk tolerance is one of the strongest dimensions separating the four clusters.

**Investment Objectives by Cluster**

Growth is primarily composed of Clusters 0 and 2, consistent with their moderate risk profiles. Aggressive Growth is almost exclusively Cluster 1, matching their high risk tolerance. Balanced Growth has the largest share from Cluster 3, reflecting a preference for diversified strategies among older, higher net worth investors. Preservation of Principal also shows a notable Cluster 3 presence, further confirming the conservative orientation of that group.

**Wealth Type Composition by Cluster**

This chart reveals the key distinction between Clusters 0 and 2. Despite sharing similar age, risk tolerance, and investment objectives, Cluster 0 is entirely Earned wealth while Cluster 2 is entirely Inherited wealth. The algorithm separated two otherwise similar young investor groups purely based on how their money was obtained. Cluster 1 is mixed with a slight Earned majority, suggesting that high risk aggressive investors behave similarly regardless of wealth source. Cluster 3 is also mixed with a slight Inherited majority, consistent with its higher net worth profile driven partly by wealth transfers.

### 8.6 Key Findings

| Cluster | Size | Avg Age | Profile |
|---------|------|---------|---------|
| **Cluster 0** | 979 (31.1%) | 27.3 | 100% earned wealth, moderate risk, growth focused, low income and net worth. Young savers building through wages and savings. |
| **Cluster 1** | 920 (29.3%) | 26.3 | Mixed wealth source (58% earned), high risk appetite, overwhelmingly aggressive growth objectives. Most volatility tolerant group. |
| **Cluster 2** | 717 (22.8%) | 28.1 | 100% inherited wealth, moderate risk, growth focused, low income and net worth. Received money via gift, inheritance, or transfer. |
| **Cluster 3** | 528 (16.8%) | 52.2 | Mixed wealth (55% inherited), low risk, balanced growth and preservation objectives. Highest income ($100K+) and net worth ($1M+). Pre retirement conservatives. |

**Cluster 0: Young Self Made Savers**

This is the largest group, consisting entirely of investors who built their wealth through their own savings and wages. They are in the early stages of their careers with relatively low income and net worth, but they show disciplined investment behavior with moderate risk tolerance and a clear focus on long term growth. Their primary funding sources are personal savings and employment income. For RBC, these investors represent a long term relationship opportunity as their income and assets are likely to grow over time.

**Cluster 1: Young Aggressive Risk Takers**

This group stands out for its high risk appetite and near universal preference for aggressive growth strategies. Unlike the other young clusters, wealth source is mixed here, meaning both self made and inherited wealth investors converge when they share a willingness to tolerate volatility. This suggests that risk attitude, rather than how the money was obtained, is the defining characteristic of this segment. These investors may respond well to offerings in equities, alternative investments, and growth oriented portfolios.

**Cluster 2: Young Passive Recipients**

This group mirrors Cluster 0 in age, risk tolerance, and investment objectives, but differs in one critical way: their wealth comes entirely from gifts, inheritance, or transfers. Despite receiving money rather than earning it, they adopt a moderate, growth oriented investment approach similar to their self made peers. However, their low personal income suggests they may be less financially experienced and more dependent on advisor guidance. This is the segment most directly connected to the Great Wealth Transfer, and proactive engagement is essential to prevent these investors from leaving RBC after receiving their inheritance.

**Cluster 3: Pre Retirement Conservatives**

This is the oldest and wealthiest group, with a median age in the mid 50s and net worth typically exceeding $1 million. They favor balanced growth and capital preservation strategies, reflecting a shift from accumulation to protection as they approach retirement. Their wealth source is mixed with a slight inherited majority, and their income levels are the highest among all four clusters. Although they are not "young" in the traditional sense, they were captured by the behavioural filter (Condition 2) due to their conservative, low risk investment profile combined with recent account activity. For RBC, this group represents the highest immediate asset value and may benefit from retirement planning and wealth preservation services.

### 8.7 Export Segmented Data

In [ ]:
segmented_output = young_investors[[
    "PartyId",
    "FinancialAccountId",
    "PartyAge_num",
    "RiskTolerance",
    "InvestmentObjectives",
    "InvestmentTimeHorizon",
    "LiquidityNeeds",
    "SourceofFunds",
    "WealthType",
    "AnnualIncome",
    "TotalNetWorth",
    "Cluster"
]]

segmented_output.to_csv("young_investors_segmented.csv", index=False)
print(f"Exported {len(segmented_output)} rows to young_investors_segmented.csv")
print(f"Columns: {list(segmented_output.columns)}")